# Integration - Model Training on Reduced Embeddings
*Evaluation of classification models trained on low-dimensional embeddings of Heart Failure Prediction Dataset*  

---

## Table of Contents
1. [Setup](#setup)  
    1.2 [Imports and Reproducibility](#imports-and-reproducibility)  
    1.2 [Data Loading](#data-loading)  

2. [Baseline (full feature space)](#baseline)
   
3. [Model Training on Dimensionality-Reduced Feature Spaces](#model-on-dim-red)
   
4. [Conclusions](#conclusions)  

<a id="setup"></a>
## Setup
---
<a id="imports-and-reproducibility"></a>
### Imports and Reproducibility

In [1]:
import os, joblib, sys, pandas as pd
sys.path.append(os.path.abspath(os.path.join('..')))

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer, f1_score
from sklearn.base import clone
from copy import deepcopy
from scripts.eda.preprocessing.pipeline import build_preprocessing_pipeline

SEED = 42

<a i="data-loading"></a>
### Data loading

In [2]:
df = joblib.load("../outputs/eda/df_cleaned.pkl")
df_copy = df.copy()

X = df_copy.iloc[:, :-1]
y = df_copy.iloc[:, -1]

TARGET_COL = 'DEATH_EVENT'

print("X shape:", X.shape)

X shape: (299, 12)


In [3]:
_full_preproc = build_preprocessing_pipeline()
preproc_no_balance = deepcopy(_full_preproc)
preproc_no_balance.steps = preproc_no_balance.steps[:-1]   # no balancing for now

X = preproc_no_balance.fit_transform(X, y)

<a id="baseline"></a>
## Baseline (full feature space)
---
Before comparing embeddings, a baseline using the full preprocessed feature space is established.

In [4]:
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
SCORING = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "f1": make_scorer(f1_score),
    "roc_auc": "roc_auc",
}

Previously optimised estimator configurations were loaded from disk. Models were cloned (fitted state removed) and refit from scratch inside cross-validation, so no pretrained parameters were reused.

In [5]:
models = {
    "RandomForest": clone(joblib.load("../outputs/models/best/randomforest.pkl")),
    "AdaBoost": clone(joblib.load("../outputs/models/best/adaboost.pkl")),
    "XGBoost": clone(joblib.load("../outputs/models/best/xgboost.pkl")),
    "Naive Bayes": clone(joblib.load("../outputs/models/best/naivebayes.pkl")),
    "SVM": clone(joblib.load("../outputs/models/best/svm.pkl")),
    "Logistic Regression": clone(joblib.load("../outputs/models/best/logisticregression.pkl")),
    "Stacking": clone(joblib.load("../outputs/models/best/STACKING.pkl")),
    "Voting": clone(joblib.load("../outputs/models/best/VOTING.pkl")),
}

In [6]:
baseline_results = []

for model_name, clf in models.items():
    pipe = Pipeline([("clf", clf)])

    scores = cross_validate(pipe, X, y, cv=CV, scoring=SCORING, n_jobs=-1)

    baseline_results.append({
        "Embedding | Model": f"Baseline | {model_name}",
        "acc_avg": float(scores["test_accuracy"].mean()),
        "balanced_acc_avg": float(scores["test_balanced_accuracy"].mean()),
        "f1_avg": float(scores["test_f1"].mean()),
        "roc_auc_avg": float(scores["test_roc_auc"].mean()),
    })

baseline_df = pd.DataFrame(baseline_results).sort_values("f1_avg", ascending=False)
baseline_df


,Embedding | Model,acc_avg,balanced_acc_avg,f1_avg,roc_auc_avg
0,Baseline | RandomForest,0.849379,0.814679,0.749115,0.909853
6,Baseline | Stacking,0.832655,0.819124,0.748515,0.904464
7,Baseline | Voting,0.836045,0.802301,0.733568,0.902342
5,Baseline | Logistic Regression,0.832655,0.792057,0.721329,0.885804
2,Baseline | XGBoost,0.825932,0.791897,0.716259,0.894357
4,Baseline | SVM,0.815932,0.773890,0.697196,0.869158
3,Baseline | Naive Bayes,0.799096,0.756512,0.666720,0.879814
1,Baseline | AdaBoost,0.782655,0.747141,0.656570,0.747141


<a id="model-on-dim-red"></a>
## Model Training on Dimensionality-Reduced Feature Spaces
---

In [7]:
embeddings = {
    "PCA": joblib.load("../outputs/dim_red/embeddings/X_pca.pkl"),
    "LDA": joblib.load("../outputs/dim_red/embeddings/X_lda.pkl"),
    "UMAP unsupervised": joblib.load("../outputs/dim_red/embeddings/X_umap_unsupervised.pkl"),
    "MIX": joblib.load("../outputs/dim_red/embeddings/X_mix.pkl"),
}

In [8]:
embedding_results = []

for emb_name, X_emb in embeddings.items():
    X_emb_eval = X_emb.to_numpy()

    for model_name, clf in models.items():
        pipe = Pipeline([("clf", clf)])

        scores = cross_validate(pipe, X_emb_eval, y, cv=CV, scoring=SCORING, n_jobs=-1)

        embedding_results.append({
            "Embedding | Model": f"{emb_name} | {model_name}",
            "acc_avg": float(scores["test_accuracy"].mean()),
            "balanced_acc_avg": float(scores["test_balanced_accuracy"].mean()),
            "f1_avg": float(scores["test_f1"].mean()),
            "roc_auc_avg": float(scores["test_roc_auc"].mean()),
        })

embedding_df = pd.DataFrame(embedding_results).sort_values("f1_avg", ascending=False)
embedding_df


,Embedding | Model,acc_avg,balanced_acc_avg,f1_avg,roc_auc_avg
24,MIX | RandomForest,0.835932,0.829589,0.757856,0.889448
30,MIX | Stacking,0.835876,0.826826,0.751726,0.882427
22,UMAP unsupervised | Stacking,0.819379,0.820298,0.748595,0.841602
31,MIX | Voting,0.835876,0.815854,0.743269,0.886196
29,MIX | Logistic Regression,0.839322,0.801996,0.735159,0.903486
3,PCA | Naive Bayes,0.835989,0.799881,0.732013,0.897287
27,MIX | Naive Bayes,0.822655,0.795205,0.724917,0.882851
11,LDA | Naive Bayes,0.829266,0.794618,0.724860,0.900913
6,PCA | Stacking,0.815876,0.795651,0.721269,0.889561
5,PCA | Logistic Regression,0.832599,0.791733,0.720937,0.893905


<a id="conclusions"></a>
## Conclusions
---

In [9]:
all_results = pd.concat([baseline_df, embedding_df])
all_results_sorted = all_results.sort_values('f1_avg', ascending=False , ignore_index=True)
all_results_sorted.head(10)


,Embedding | Model,acc_avg,balanced_acc_avg,f1_avg,roc_auc_avg
0,MIX | RandomForest,0.835932,0.829589,0.757856,0.889448
1,MIX | Stacking,0.835876,0.826826,0.751726,0.882427
2,Baseline | RandomForest,0.849379,0.814679,0.749115,0.909853
3,UMAP unsupervised | Stacking,0.819379,0.820298,0.748595,0.841602
4,Baseline | Stacking,0.832655,0.819124,0.748515,0.904464
5,MIX | Voting,0.835876,0.815854,0.743269,0.886196
6,MIX | Logistic Regression,0.839322,0.801996,0.735159,0.903486
7,Baseline | Voting,0.836045,0.802301,0.733568,0.902342
8,PCA | Naive Bayes,0.835989,0.799881,0.732013,0.897287
9,MIX | Naive Bayes,0.822655,0.795205,0.724917,0.882851
